# Supervisor Chat Demo (DBQNA / RAG / FAQ) – Notebook

Gunakan notebook ini untuk menjalankan router supervisor tanpa Streamlit. Pastikan `.env` sudah diisi `GOOGLE_API_KEY` dan `DB_PATH`.

In [12]:
!D:\Dokumentasi\revou-gen-ai-tutorial-main\.venv\Scripts\python.exe -m pip install --upgrade pip
!D:\Dokumentasi\revou-gen-ai-tutorial-main\.venv\Scripts\python.exe -m pip install --no-cache-dir -r D:\Dokumentasi\revou-gen-ai-tutorial-main\requirements-supervisor.txt

In [13]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv

# Cari root repo secara dinamis
repo_root = Path.cwd().resolve()
if not (repo_root / "requirements-supervisor.txt").exists():
    candidate = Path(r"D:/Dokumentasi/revou-gen-ai-tutorial-main")
    if candidate.exists() and (candidate / "requirements-supervisor.txt").exists():
        repo_root = candidate
os.chdir(repo_root)

# Simpan cache model di folder repo, bukan default user profile di C:\
hf_cache = repo_root / ".hf-cache"
os.environ.setdefault("HF_HOME", str(hf_cache))
os.environ.setdefault("SENTENCE_TRANSFORMERS_HOME", str(hf_cache / "sentence-transformers"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(hf_cache / "transformers"))

# Default untuk API Ollama (OpenAI-compatible)
os.environ.setdefault("OPENAI_BASE_URL", "http://localhost:11434/v1")
os.environ.setdefault("OPENAI_API_KEY", "ollama")

def detect_ollama_model(default: str = "llama3.2:1b") -> str:
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=False)
    except FileNotFoundError:
        return default

    installed = []
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            installed.append(parts[0])

    preferred = ["llama3.2:1b", "qwen2.5:1.5b", "gemma3:latest", "llama3"]
    for name in preferred:
        if name in installed:
            return name
    return installed[0] if installed else default

os.environ.setdefault("OLLAMA_MODEL", detect_ollama_model())

load_dotenv(override=True)
print("Repo root:", repo_root)
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("OLLAMA_MODEL:", os.getenv("OLLAMA_MODEL"))
print("HF_HOME:", os.getenv("HF_HOME"))


Repo root: D:\Dokumentasi\revou-gen-ai-tutorial-main
OPENAI_BASE_URL: http://localhost:11434/v1
OLLAMA_MODEL: gemma3:latest
HF_HOME: D:\Dokumentasi\revou-gen-ai-tutorial-main\.hf-cache


In [14]:
import os, importlib, agents.FAQ, agents.RAG, agents.DBQNA, agents.graph
for m in [agents.FAQ, agents.RAG, agents.DBQNA, agents.graph]:
    importlib.reload(m)
import os
print("GEMINI_MODEL env:", os.getenv("GEMINI_MODEL"))


GEMINI_MODEL env: gemini-2.0-flash


In [15]:
import os, pathlib
print("CWD:", pathlib.Path.cwd())
from dotenv import load_dotenv; load_dotenv(override=True)
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("DB_PATH:", os.getenv("DB_PATH"))


CWD: D:\Dokumentasi\revou-gen-ai-tutorial-main
OPENAI_BASE_URL: http://localhost:11434/v1
DB_PATH: D:\path\ke\sqlite.db


In [16]:
import os
from pathlib import Path
# cari root repo secara dinamis
repo_root = Path.cwd().resolve()
if not (repo_root / "requirements-supervisor.txt").exists():
    candidate = Path(r"D:/Dokumentasi/revou-gen-ai-tutorial-main")
    if candidate.exists() and (candidate / "requirements-supervisor.txt").exists():
        repo_root = candidate
    else:
        raise FileNotFoundError("requirements-supervisor.txt tidak ditemukan. Buka notebook dari folder repo yang benar.")
os.chdir(repo_root)
req_path = repo_root / "requirements-supervisor.txt"
req_path


WindowsPath('D:/Dokumentasi/revou-gen-ai-tutorial-main/requirements-supervisor.txt')

In [17]:
# (Opsional) install dependensi minimal untuk notebook ini jika belum
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-r", str(req_path)], check=False)
print("Interpreter aktif:", sys.executable)
print("Requirements:", req_path)

Interpreter aktif: d:\Dokumentasi\revou-gen-ai-tutorial-main\.venv\Scripts\python.exe
Requirements: D:\Dokumentasi\revou-gen-ai-tutorial-main\requirements-supervisor.txt


In [18]:
# Muat environment
from dotenv import load_dotenv
load_dotenv(override=True)
import os
from pathlib import Path

assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL belum diset (contoh: http://localhost:11434/v1)"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY belum diset (isi bebas, default 'ollama')"

repo_root = Path.cwd().resolve()
db_candidates = []
if os.getenv("DB_PATH"):
    db_candidates.append(Path(os.environ["DB_PATH"]))
db_candidates.extend([
    repo_root / "sqlite" / "chinook.db",
    repo_root / "sqlite" / "test.db",
])

resolved_db_path = next((p for p in db_candidates if p.exists()), None)
assert resolved_db_path is not None, "Tidak ada database yang ditemukan. Periksa DB_PATH atau folder sqlite/."
if os.getenv("DB_PATH") != str(resolved_db_path):
    os.environ["DB_PATH"] = str(resolved_db_path)
    print(f"DB_PATH tidak valid atau kosong. Menggunakan fallback: {resolved_db_path}")

tracing_enabled = os.getenv("LANGCHAIN_TRACING_V2", "false").lower() in {"1", "true", "yes"} or os.getenv("LANGSMITH_TRACING", "false").lower() in {"1", "true", "yes"}
langsmith_key = os.getenv("LANGCHAIN_API_KEY") or os.getenv("LANGSMITH_API_KEY")
if tracing_enabled and not langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing dimatikan karena API key belum diset.")

print("Env OK")
print("DB_PATH:", os.getenv("DB_PATH"))
print("OLLAMA_MODEL:", os.getenv("OLLAMA_MODEL"))
print("HF_HOME:", os.getenv("HF_HOME"))
print("LangSmith tracing:", os.getenv("LANGCHAIN_TRACING_V2") or os.getenv("LANGSMITH_TRACING"))


DB_PATH tidak valid atau kosong. Menggunakan fallback: D:\Dokumentasi\revou-gen-ai-tutorial-main\sqlite\chinook.db
Env OK
DB_PATH: D:\Dokumentasi\revou-gen-ai-tutorial-main\sqlite\chinook.db
OLLAMA_MODEL: gemma3:latest
HF_HOME: D:\Dokumentasi\revou-gen-ai-tutorial-main\.hf-cache
LangSmith tracing: true


In [19]:
# Pastikan index FAQ tersedia
from agents.FAQ import build_vectorstore
build_vectorstore()
print("FAQ vectorstore siap")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

FAQ vectorstore siap


In [20]:
# Definisi supervisor graph (mirip Streamlit)
import os
from uuid import uuid4
from typing import Literal
from openai import BadRequestError
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.types import Command

import agents.DBQNA as DBQNA
import agents.RAG as RAG
import agents.FAQ as FAQ

FAQ_THREAD_ID = f"faq-notebook-{uuid4()}"

model = ChatOpenAI(
    model=os.getenv("OLLAMA_MODEL", "llama3.2:1b"),
    base_url=os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1"),
    api_key=os.getenv("OPENAI_API_KEY", "ollama"),
    temperature=0,
    model_kwargs={"response_format": {"type": "json_object"}},
)
DB_PATH = os.environ["DB_PATH"]

class BestAgent(BaseModel):
    agent_name: Literal["DBQNA", "RAG", "FAQ"] = Field(description="Agent terbaik untuk pertanyaan ini.")

class SupervisorState(MessagesState):
    user_question: str

def supervisor(state: SupervisorState) -> Command[Literal["DBQNA", "RAG", "FAQ", END]]:
    last_message = state["messages"][-1]
    instruction = [
        SystemMessage(
            content="""Pilih agen yang tepat.
            DBQNA: pertanyaan SQL/database.
            RAG: profil perusahaan Dexa Medica.
            FAQ: layanan pelanggan/FAQ (kontak, jam, pemesanan, pengaduan, kebijakan)."""
        )
    ]
    model_with_structure = model.with_structured_output(BestAgent)
    response = model_with_structure.invoke(instruction + [last_message])
    return Command(update={"user_question": last_message.content}, goto=response.agent_name)

def callRAG(state: SupervisorState):
    prompt = state["user_question"]
    response = RAG.graph.invoke({"messages": [HumanMessage(content=prompt)]})
    return Command(goto=END, update={"messages": response["messages"][-1]})

def callDBQNA(state: SupervisorState):
    prompt = state["user_question"]
    try:
        response = DBQNA.graph.invoke({"messages": [HumanMessage(content=prompt)], "db_name": DB_PATH, "user_question": prompt})
    except BadRequestError as exc:
        if "does not support tools" in str(exc):
            message = AIMessage(content="Model Ollama yang aktif tidak mendukung tool calling untuk agent DBQNA. Gunakan model lain untuk pertanyaan database, misalnya model Ollama yang mendukung tools.")
            return Command(goto=END, update={"messages": message})
        raise
    return Command(goto=END, update={"messages": response["messages"][-1]})

def callFAQ(state: SupervisorState):
    prompt = state["user_question"]
    response = FAQ.graph.invoke(
        {"messages": [HumanMessage(content=prompt)], "user_question": prompt, "attempt": 0, "rewritten_question": ""},
        config={"configurable": {"thread_id": FAQ_THREAD_ID}},
    )
    return Command(goto=END, update={"messages": response["messages"][-1]})

supervisor_graph = (
    StateGraph(SupervisorState)
    .add_node(supervisor)
    .add_node("RAG", callRAG)
    .add_node("DBQNA", callDBQNA)
    .add_node("FAQ", callFAQ)
    .add_edge(START, "supervisor")
    .compile(name="supervisor-notebook")
)
print("Supervisor graph siap. FAQ thread:", FAQ_THREAD_ID)


Supervisor graph siap. FAQ thread: faq-notebook-606c4771-9e7d-4aa6-aac0-991282712c31


In [21]:
import os
import subprocess

print("OLLAMA_MODEL:", os.getenv("OLLAMA_MODEL"))
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))

try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=False)
    print(result.stdout.strip())
except FileNotFoundError:
    print("Perintah 'ollama' tidak ditemukan di PATH.")


OLLAMA_MODEL: gemma3:latest
OPENAI_BASE_URL: http://localhost:11434/v1
NAME                  ID              SIZE      MODIFIED    
minimax-m2.5:cloud    c0d5751c800f    -         2 hours ago    
gemma3:latest         a2af6cc3eb7f    3.3 GB    3 hours ago


In [23]:
# Contoh pemanggilan
from langchain_core.messages import HumanMessage
from openai import NotFoundError

question = "Dimana saja pabrik Dexa Medica?"

try:
    result = supervisor_graph.invoke({"messages": [HumanMessage(content=question)]})
    final_message = result["messages"] if hasattr(result["messages"], "content") else result["messages"][-1]
    print("Final agent reply:\n", final_message.content)
except NotFoundError as exc:
    print("Model Ollama tidak ditemukan. Periksa OLLAMA_MODEL di .env dan jalankan cell diagnostik `ollama list`.")
    print(exc)


Final agent reply:
 Dexa Medica memiliki tiga pabrik utama: PT DEXA MEDICA PALEMBANG, PT DEXA MEDICA CIKARANG, dan PT BETA PHARMACON KARAWANG.
